## 1. Data Loading

In [ ]:
import pandas as pd

df = pd.read_csv("propertyfinder_final.csv")

## 2. Initial Data Exploration

In [ ]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39712 entries, 0 to 39711
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   listing_id         39712 non-null  object 
 1   category           39712 non-null  object 
 2   listing_type       39712 non-null  object 
 3   property_type      39712 non-null  object 
 4   completion_status  39712 non-null  object 
 5   price_egp          39712 non-null  int64  
 6   price_period       39712 non-null  object 
 7   city               39712 non-null  object 
 8   town               39712 non-null  object 
 9   district           39712 non-null  object 
 10  subdistrict        39712 non-null  object 
 11  lat                39712 non-null  float64
 12  lon                39712 non-null  float64
 13  bedrooms           39712 non-null  int64  
 14  bathrooms          39712 non-null  int64  
 15  area_value         39712 non-null  int64  
 16  furnished          397

,0
listing_id,0
category,0
listing_type,0
property_type,0
completion_status,0
price_egp,0
price_period,0
city,0
town,0
district,0


## 3. Feature Engineering

In [ ]:
df["price_per_sqm"] = df["price_egp"] / df["area_value"]

### Price Per Square Meter

In [ ]:
df["total_rooms"] = df["bedrooms"] + df["bathrooms"]

### Total Rooms

In [ ]:
city_popularity = df["city"].value_counts(normalize=True)

df["location_popularity"] = df["city"].map(city_popularity)

### Location Popularity Score

In [ ]:
df["is_premium"] = (
    df["listing_level"]=="Premium"
).astype(int)

### Is Premium Listing

In [ ]:
df["price_score"] = (
    1 - (
        (df["price_per_sqm"] - df["price_per_sqm"].min())
        /
        (df["price_per_sqm"].max() - df["price_per_sqm"].min())
    )
)

### Normalized Price Score

In [ ]:
df["location_score"] = (
    df["location_popularity"]*10
)

### Scaled Location Score

In [ ]:
df["investment_score"] = (
      0.4 * df["price_score"]
    + 0.3 * df["location_score"]
    + 0.2 * df["is_premium"]
    + 0.1 * df["images_count"]
)

### Composite Investment Score Calculation

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler((0,10))

df["investment_score"] = scaler.fit_transform(
    df[["investment_score"]]
)

### Scale Investment Score to 0-10

In [ ]:
features = [
    "price_egp",
    "area_value",
    "bedrooms",
    "bathrooms",
    "price_per_sqm",
    "location_popularity"
]

## 4. Prepare Data for Modeling

In [ ]:
target = "investment_score"

### One-Hot Encoding Categorical Features

In [ ]:
df = pd.get_dummies(
    df,
    columns=[
        "category",
        "listing_type",
        "completion_status",
        "price_period",
        "furnished",
        "listing_level"
    ]
)

### Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(
    [
        "investment_score",
        "listing_id",
        "listed_date",
        "town",
        "district",
        "subdistrict",
        "broker_name"
    ],
    axis=1
)

y = df["investment_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# The previous model fit failed because some columns were still non-numeric.
# Let's re-define X and Y to ensure all non-numeric columns are handled.
# This cell replaces the previous `uHA2zxG7hqzL` content by ensuring all
# necessary columns are dropped or encoded.
from sklearn.model_selection import train_test_split

X = df.drop(
    [
        "investment_score",
        "listing_id",
        "listed_date",
        "town",
        "district",
        "subdistrict",
        "broker_name",
        "agent_id", # dropping agent_id as it's an identifier
        "broker_id", # dropping broker_id as it's an identifier
        # is_featured and has_view_360 are boolean and will be handled fine by default
    ],
    axis=1
)

y = df["investment_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 5. Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

model.fit(X_train,y_train)

RandomForestRegressor(n_estimators=300, random_state=42)

## 6. Model Evaluation

In [ ]:
from sklearn.metrics import r2_score

pred = model.predict(X_test)

r2_score(y_test,pred)

0.9999684772811753

## 7. Prediction and Categorization

In [ ]:
df["predicted_investment_score"] = model.predict(X)

### Categorize Investment Level

In [ ]:
df["investment_level"] = pd.cut(
    df["predicted_investment_score"],
    bins=[0,4,7,10],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

## 8. Save Results

In [ ]:
df.to_csv(
    "investment_scored_properties.csv",
    index=False
)